
# Hyperparameter-Optimierung (Server) – RF, LSTM, XGBoost, LightGBM, CNN1D

**Wichtig:** Optimierung läuft *immer auf dem Server*.  
**Deployment-Profile:** *edge* (schlanker Suchraum, kein FE) · *server* (größerer Suchraum, **rolling mean + std**).

- Zielvariable: `Group4-2_S6_VolumetricFlowRate`
- `lags`/`horizon`: **fix** (nicht Teil der Optimierung)
- **Zeitspalten** sind **keine** Features.
- **Feature Importance (Permutation, gruppiert)** über 6 Basissignale:
  - `Group4-2_S6_MassFlowRate`
  - `Group4-2_S6_FlowVelocity`
  - `Group4-2_S6_Volume`
  - `Group4-2_S6_VolumetricFlowRate` *(Vergangenheitsziel OK)*
  - `Group4-2_S6_Temperature`
  - `Group4-2_S6_Pressure`
  → Nur **sinnvolle** Signale wandern in die **Server-Features** (mit rolling mean/std). Das *Edge*-Profil nutzt dieselben sinnvollen Signale, aber **ohne** FE.


In [ ]:

import os, json, math, random, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

# ---- Settings ----
DATA_PATH = r"""C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_rate_limited.csv"""
TARGET_COL = "Group4-2_S6_VolumetricFlowRate"
EXCLUDE_COLS = ["time", "datetime", "recording_timestamp", "duration"]

LAGS = 4
HORIZON = 4

TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.2  # für DL-Validierung

CANDIDATE_BASE_FEATURES = [
    "Group4-2_S6_MassFlowRate",
    "Group4-2_S6_FlowVelocity",
    "Group4-2_S6_Volume",
    "Group4-2_S6_VolumetricFlowRate",
    "Group4-2_S6_Temperature",
    "Group4-2_S6_Pressure",
]


In [ ]:

def display_dataframe_to_user(name, df):
    try:
        fn = f"{name}.csv".replace(" ", "_")
        df.to_csv(fn, index=False)
        print(f"[Saved] {fn}")
    except Exception as e:
        print("Could not save CSV:", e)
    try:
        display(df.head(20))
    except Exception:
        print(df.head(20))


## 1) Daten laden & Sichten

In [ ]:

df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
if "datetime" in df.columns:
    try:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.sort_values("datetime").set_index("datetime")
    except Exception:
        pass

print("Shape:", df.shape)
display(df.head(3))

nulls = df.isna().mean().sort_values(ascending=False)
print("\nMissing-Rate (Top 10):")
display(nulls.head(10).to_frame("missing_rate"))

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cols_to_drop = set([c for c in EXCLUDE_COLS if c in df.columns])
numeric_features = [c for c in num_cols if c not in cols_to_drop and c != TARGET_COL]

print(f"\nNumerische Features (ohne Zeit & Target): {len(numeric_features)}")
print(numeric_features[:20])

if isinstance(df.index, pd.DatetimeIndex) and TARGET_COL in df.columns:
    plt.figure(figsize=(12,4))
    df[TARGET_COL].plot(); plt.title("Zielvariable über Zeit"); plt.show()

if TARGET_COL in df.columns:
    plt.figure(figsize=(6,4))
    df[TARGET_COL].plot(kind="hist", bins=50); plt.title("Histogramm Zielvariable"); plt.show()


## 2) Feature Engineering (Profile: EDGE vs SERVER)

In [ ]:

def build_features(df_in, base_feats, target_col, *, enable=False, roll_windows=None, stats=None,
                   add_diff=False, add_pct=False, ewm_alphas=None):
    if not enable:
        return df_in.copy(), base_feats.copy()
    roll_windows = roll_windows or []
    stats = stats or []
    ewm_alphas = ewm_alphas or []
    df_out = df_in.copy()
    cols = list(set(base_feats + [target_col]))
    for col in cols:
        s = df_out[col]
        for w in roll_windows:
            r = s.rolling(window=w, min_periods=w)
            if "mean" in stats: df_out[f"{col}_roll{w}_mean"] = r.mean()
            if "std"  in stats: df_out[f"{col}_roll{w}_std"]  = r.std()
        if add_diff: df_out[f"{col}_diff1"] = s.diff(1)
        if add_pct:  df_out[f"{col}_pct1"]  = s.pct_change(1)
        for a in ewm_alphas:
            df_out[f"{col}_ewm{str(a).replace('.','_')}"] = s.ewm(alpha=a, adjust=False).mean()
    df_out = df_out.dropna()
    num_cols_fe = df_out.select_dtypes(include=[np.number]).columns.tolist()
    active = [c for c in num_cols_fe if c not in set(EXCLUDE_COLS) and c != target_col]
    return df_out, active

FE_EDGE = dict(enable=False, roll_windows=[], stats=[], add_diff=False, add_pct=False, ewm_alphas=[])
FE_SERVER = dict(enable=True,  roll_windows=[3,5,9,15], stats=["mean","std"], add_diff=False, add_pct=False, ewm_alphas=[])


## 3) Supervised Datensatz (mit Spaltennamen)

In [ ]:

def make_supervised_df(df_in, features, target, lags, horizon):
    data = df_in[features + [target]].dropna().copy()
    F = len(features)
    X_rows = []
    col_names = []
    for lag in range(lags, 0, -1):
        for feat in features:
            col_names.append(f"{feat}|t-{lag}")
    Y = []
    vals = data.values
    for i in range(lags, len(data) - horizon + 1):
        past = []
        for lag in range(lags, 0, -1):
            past.extend(vals[i-lag, :F])
        X_rows.append(past)
        Y.append(vals[i:i+horizon, F])
    X2d_df = pd.DataFrame(np.array(X_rows), columns=col_names)
    Y = np.array(Y)
    return X2d_df, Y

def split_train_test(X, y, frac):
    n = len(X); s = int(n*frac)
    return X[:s], y[:s], X[s:], y[s:]

def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred).reshape(len(y_pred), -1)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return dict(mae=mae, rmse=rmse, mse=mse, r2=r2)


## 4) Grouped Permutation Feature Importance (Server-Profil)

In [ ]:

base_server_candidates = [c for c in CANDIDATE_BASE_FEATURES if c in df.columns]
df_srv_fe, active_srv = build_features(df, base_server_candidates, TARGET_COL, **FE_SERVER)

X2d_srv, Y_srv = make_supervised_df(df_srv_fe, active_srv, TARGET_COL, LAGS, HORIZON)
X2d_srv_tr, Y_srv_tr, X2d_srv_te, Y_srv_te = split_train_test(X2d_srv.values, Y_srv, TRAIN_FRACTION)

rf_imp = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=600, max_depth=None, min_samples_split=4, min_samples_leaf=2,
    max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1
))
rf_imp.fit(X2d_srv_tr, Y_srv_tr)
pred0 = rf_imp.predict(X2d_srv_te)
base = eval_metrics(Y_srv_te, pred0)
print("Baseline RF (Server-FE) overall:", base)

# group permutation
if isinstance(X2d_srv, pd.DataFrame):
    cols = list(X2d_srv.columns)
else:
    cols = [f"c{i}" for i in range(X2d_srv.shape[1])]
    X2d_srv = pd.DataFrame(X2d_srv, columns=cols)

def group_mask(columns, signal):
    pat = re.compile(rf"^{re.escape(signal)}(\||_)")
    return [i for i, c in enumerate(columns) if pat.match(str(c))]

results = []
rng = np.random.default_rng(123)
for sig in base_server_candidates:
    idxs = group_mask(cols, sig)
    if not idxs: continue
    Xp = X2d_srv.copy()
    for j in idxs:
        Xp.iloc[:, j] = rng.permutation(Xp.iloc[:, j].values)
    pred = rf_imp.predict(Xp.values)
    m = eval_metrics(Y_srv_te, pred)
    imp_rmse = m["rmse"] - base["rmse"]
    rel = (imp_rmse / base["rmse"] * 100.0) if base["rmse"]>0 else 0.0
    results.append(dict(signal=sig, rmse_increase=imp_rmse, rel_percent=rel, r2_drop=base["r2"]-m["r2"]))

imp_df = pd.DataFrame(results).sort_values(["rmse_increase"], ascending=False)
display_dataframe_to_user("Server_GroupPermutation_Importance", imp_df)

sensible = imp_df[imp_df["rel_percent"] > 1.0]["signal"].tolist()
print("Sinnvolle Basis-Signale (Server):", sensible)

# final datasets
if sensible:
    df_srv_final, active_srv_final = build_features(df, sensible, TARGET_COL, **FE_SERVER)
else:
    df_srv_final, active_srv_final = df_srv_fe, active_srv

df_edge_final, active_edge_final = build_features(df, sensible if sensible else base_server_candidates, TARGET_COL, enable=False)
print(f"Server-Features: {len(active_srv_final)} | Edge-Features: {len(active_edge_final)}")


## 5) Datensätze bauen (EDGE & SERVER, final)

In [ ]:

X2d_srv_final, Y_srv_final = make_supervised_df(df_srv_final, active_srv_final, TARGET_COL, LAGS, HORIZON)
X2d_srv_tr, Y_srv_tr, X2d_srv_te, Y_srv_te = split_train_test(X2d_srv_final.values, Y_srv_final, TRAIN_FRACTION)

X2d_edge_final, Y_edge_final = make_supervised_df(df_edge_final, active_edge_final, TARGET_COL, LAGS, HORIZON)
X2d_edge_tr, Y_edge_tr, X2d_edge_te, Y_edge_te = split_train_test(X2d_edge_final.values, Y_edge_final, TRAIN_FRACTION)

print("SERVER shapes: X_tr", X2d_srv_tr.shape, "Y_tr", Y_srv_tr.shape, "| X_te", X2d_srv_te.shape, "Y_te", Y_srv_te.shape)
print("EDGE   shapes: X_tr", X2d_edge_tr.shape, "Y_tr", Y_edge_tr.shape, "| X_te", X2d_edge_te.shape, "Y_te", Y_edge_te.shape)


## 6) Configs (mit FE-Block) im gleichen Ordner erstellen/warten

In [ ]:

BASE = {
    "loading_strategy": "split",
    "train_fraction": 0.8,
    "validation_fraction": 0.2,
    "dataset": DATA_PATH,
    "target_column": TARGET_COL,
    "exclude_columns": EXCLUDE_COLS,
    "lags": LAGS,
    "horizon": HORIZON,
    "base_features": CANDIDATE_BASE_FEATURES,
    "optimize_on": "server",
}

FE_EDGE_JSON = {"enable": False, "rolling_windows": [], "stats": [], "add_diff": False, "add_pct": False, "ewm_alphas": []}
FE_SERVER_JSON = {"enable": True, "rolling_windows": [3,5,9,15], "stats": ["mean","std"], "add_diff": False, "add_pct": False, "ewm_alphas": []}

# search spaces (wie zuvor) -- RF/LSTM/XGB/LGBM/CNN
RF_EDGE = {"n_estimators":{"type":"int","min":50,"max":250},"max_depth":{"type":"int_or_none","min":5,"max":20},
           "min_samples_split":{"type":"int","min":2,"max":10},"min_samples_leaf":{"type":"int","min":1,"max":8},
           "max_features":{"type":"float","min":0.3,"max":0.9},"bootstrap":{"type":"choice","choices":[True,False]},
           "n_jobs":{"type":"fixed","value":-1},"random_state":{"type":"fixed","value":42}}
RF_SERVER = {"n_estimators":{"type":"int","min":200,"max":1000},"max_depth":{"type":"int_or_none","min":10,"max":40},
             "min_samples_split":{"type":"int","min":2,"max":20},"min_samples_leaf":{"type":"int","min":1,"max":10},
             "max_features":{"type":"float","min":0.3,"max":1.0},"bootstrap":{"type":"choice","choices":[True,False]},
             "n_jobs":{"type":"fixed","value":-1},"random_state":{"type":"fixed","value":42}}

LSTM_EDGE = {"num_layers":{"type":"fixed","value":1},"initial_units":{"type":"int","min":18,"max":64},"dropout":{"type":"float","min":0.0,"max":0.35},
             "batch_size":{"type":"int_choice","choices":[16,24,32,48,64]},"epochs":{"type":"int","min":20,"max":60},
             "learning_rate":{"type":"log_float","min":1e-4,"max":5e-3},"loss":{"type":"choice","choices":["mse","mae","huber"]},
             "optimizer":{"type":"choice","choices":["adam","rmsprop","nadam"]},"clipnorm":{"type":"float","min":0.5,"max":3.0}}
LSTM_SERVER = {"num_layers":{"type":"int","min":2,"max":3},"initial_units":{"type":"int","min":56,"max":128},"dropout":{"type":"float","min":0.05,"max":0.45},
               "batch_size":{"type":"int_choice","choices":[32,48,64,96,128]},"epochs":{"type":"int","min":50,"max":120},
               "learning_rate":{"type":"log_float","min":1e-4,"max":3e-3},"loss":{"type":"choice","choices":["mse","mae","huber"]},
               "optimizer":{"type":"choice","choices":["adam","rmsprop","nadam","adamw"]},"clipnorm":{"type":"float","min":0.5,"max":5.0},
               "weight_decay":{"type":"log_float","min":1e-6,"max":1e-3}}

XGB_EDGE = {"n_estimators":{"type":"int","min":100,"max":400},"max_depth":{"type":"int","min":3,"max":8},
            "learning_rate":{"type":"log_float","min":0.01,"max":0.2},"subsample":{"type":"float","min":0.6,"max":1.0},
            "colsample_bytree":{"type":"float","min":0.6,"max":1.0},"min_child_weight":{"type":"int","min":1,"max":8},
            "gamma":{"type":"float","min":0.0,"max":5.0},"reg_lambda":{"type":"log_float","min":1e-3,"max":10.0},
            "reg_alpha":{"type":"log_float","min":1e-6,"max":1e-1},"tree_method":{"type":"fixed","value":"hist"},
            "n_jobs":{"type":"fixed","value":-1},"random_state":{"type":"fixed","value":42}}
XGB_SERVER = {"n_estimators":{"type":"int","min":400,"max":1500},"max_depth":{"type":"int","min":6,"max":12},
              "learning_rate":{"type":"log_float","min":0.01,"max":0.1},"subsample":{"type":"float","min":0.6,"max":1.0},
              "colsample_bytree":{"type":"float","min":0.6,"max":1.0},"min_child_weight":{"type":"int","min":1,"max":12},
              "gamma":{"type":"float","min":0.0,"max":8.0},"reg_lambda":{"type":"log_float","min":1e-3,"max":100.0},
              "reg_alpha":{"type":"log_float","min":1e-6,"max":1.0},"tree_method":{"type":"fixed","value":"hist"},
              "n_jobs":{"type":"fixed","value":-1},"random_state":{"type":"fixed","value":42}}

LGBM_EDGE = {"n_estimators":{"type":"int","min":200,"max":1000},"learning_rate":{"type":"log_float","min":0.01,"max":0.2},
             "num_leaves":{"type":"int","min":16,"max":64},"min_child_samples":{"type":"int","min":5,"max":50},
             "subsample":{"type":"float","min":0.6,"max":1.0},"colsample_bytree":{"type":"float","min":0.6,"max":1.0},
             "reg_alpha":{"type":"log_float","min":1e-6,"max":1e-1},"reg_lambda":{"type":"log_float","min":1e-3,"max":10.0},
             "max_bin":{"type":"int","min":63,"max":255}}
LGBM_SERVER = {"n_estimators":{"type":"int","min":500,"max":3000},"learning_rate":{"type":"log_float","min":0.01,"max":0.1},
               "num_leaves":{"type":"int","min":31,"max":255},"min_child_samples":{"type":"int","min":5,"max":100},
               "subsample":{"type":"float","min":0.5,"max":1.0},"colsample_bytree":{"type":"float","min":0.5,"max":1.0},
               "reg_alpha":{"type":"log_float","min":1e-6,"max":1.0},"reg_lambda":{"type":"log_float","min":1e-3,"max":100.0},
               "max_bin":{"type":"int","min":127,"max":511}}

CNN_EDGE = {"cnn_blocks":{"type":"int","min":1,"max":2},"cnn_base_filters":{"type":"int","min":16,"max":64},
            "cnn_kernel_size":{"type":"int","min":3,"max":7},"cnn_dropout":{"type":"float","min":0.0,"max":0.35},
            "cnn_activation":{"type":"choice","choices":["relu","gelu"]},"batch_size":{"type":"int_choice","choices":[16,32,48,64]},
            "epochs":{"type":"int","min":20,"max":60},"optimizer":{"type":"choice","choices":["adam","rmsprop","nadam"]},
            "learning_rate":{"type":"log_float","min":1e-4,"max":5e-3},"clipnorm":{"type":"float","min":0.5,"max":3.0}}
CNN_SERVER = {"cnn_blocks":{"type":"int","min":2,"max":4},"cnn_base_filters":{"type":"int","min":64,"max":256},
              "cnn_kernel_size":{"type":"int","min":3,"max":9},"cnn_dropout":{"type":"float","min":0.05,"max":0.5},
              "cnn_activation":{"type":"choice","choices":["relu","gelu"]},"batch_size":{"type":"int_choice","choices":[32,64,96,128]},
              "epochs":{"type":"int","min":50,"max":150},"optimizer":{"type":"choice","choices":["adam","rmsprop","nadam","adamw"]},
              "learning_rate":{"type":"log_float","min":1e-4,"max":3e-3},"clipnorm":{"type":"float","min":0.5,"max":5.0},
              "weight_decay":{"type":"log_float","min":1e-6,"max":1e-3}}

CONFIG_TEMPLATES = {
    "config_rf_edge_opt.json":   {**BASE, "model_name":"random_forest_edge_opt",  "deployment_profile":"edge",   "feature_engineering":FE_EDGE_JSON,   "search_space":RF_EDGE},
    "config_rf_server_opt.json": {**BASE, "model_name":"random_forest_server_opt","deployment_profile":"server", "feature_engineering":FE_SERVER_JSON, "search_space":RF_SERVER},
    "config_lstm_edge_opt.json": {**BASE, "model_name":"lstm_edge_opt",           "deployment_profile":"edge",   "feature_engineering":FE_EDGE_JSON,   "search_space":LSTM_EDGE},
    "config_lstm_server_opt.json":{**BASE,"model_name":"lstm_server_opt",         "deployment_profile":"server", "feature_engineering":FE_SERVER_JSON, "search_space":LSTM_SERVER},
    "config_xgb_edge_opt.json":  {**BASE, "model_name":"xgboost_edge_opt",        "deployment_profile":"edge",   "feature_engineering":FE_EDGE_JSON,   "search_space":XGB_EDGE},
    "config_xgb_server_opt.json":{**BASE, "model_name":"xgboost_server_opt",      "deployment_profile":"server", "feature_engineering":FE_SERVER_JSON, "search_space":XGB_SERVER},
    "config_lgbm_edge_opt.json": {**BASE, "model_name":"lightgbm_edge_opt",       "deployment_profile":"edge",   "feature_engineering":FE_EDGE_JSON,   "search_space":LGBM_EDGE},
    "config_lgbm_server_opt.json":{**BASE,"model_name":"lightgbm_server_opt",     "deployment_profile":"server", "feature_engineering":FE_SERVER_JSON, "search_space":LGBM_SERVER},
    "config_cnn1d_edge_opt.json":{**BASE, "model_name":"cnn1d_edge_opt",          "deployment_profile":"edge",   "feature_engineering":FE_EDGE_JSON,   "search_space":CNN_EDGE},
    "config_cnn1d_server_opt.json":{**BASE,"model_name":"cnn1d_server_opt",       "deployment_profile":"server", "feature_engineering":FE_SERVER_JSON, "search_space":CNN_SERVER},
}

for fname, payload in CONFIG_TEMPLATES.items():
    if not os.path.exists(fname):
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        print("[created]", fname)
    else:
        print("[exists ]", fname)


## 7) Random Forest – Random Search (EDGE & SERVER)

In [ ]:

def sample_from_space(space):
    out = {}
    for k, spec in space.items():
        t = spec.get("type")
        if t == "fixed": out[k] = spec["value"]
        elif t == "int": out[k] = int(np.random.randint(spec["min"], spec["max"] + 1))
        elif t == "float": out[k] = float(np.random.uniform(spec["min"], spec["max"]))
        elif t == "log_float":
            lo, hi = math.log(spec["min"]), math.log(spec["max"])
            out[k] = float(math.exp(np.random.uniform(lo, hi)))
        elif t == "int_choice": out[k] = int(np.random.choice(spec["choices"]))
        elif t == "choice": out[k] = random.choice(spec["choices"])
        elif t == "int_or_none":
            val = int(np.random.randint(spec["min"], spec["max"] + 1))
            out[k] = None if np.random.rand() < 0.1 else val
        else: raise ValueError(f"Unsupported type: {t}")
    return out

with open("config_rf_edge_opt.json","r",encoding="utf-8") as f: rf_edge_cfg = json.load(f)
with open("config_rf_server_opt.json","r",encoding="utf-8") as f: rf_server_cfg = json.load(f)

def rf_random_search(space, X_tr, Y_tr, X_te, Y_te, trials=40):
    results, best = [], None
    for _ in range(trials):
        p = sample_from_space(space)
        rf = RandomForestRegressor(**{k:p[k] for k in p if k in [
            "n_estimators","max_depth","min_samples_split","min_samples_leaf","max_features","bootstrap","random_state","n_jobs"
        ]})
        rf.fit(X_tr, Y_tr)
        pred = rf.predict(X_te)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]: best = row
    return pd.DataFrame(results), best

df_rf_edge, best_rf_edge = rf_random_search(rf_edge_cfg["search_space"], X2d_edge_tr, Y_edge_tr, X2d_edge_te, Y_edge_te, trials=30)
df_rf_server, best_rf_server = rf_random_search(rf_server_cfg["search_space"], X2d_srv_tr, Y_srv_tr, X2d_srv_te, Y_srv_te, trials=50)

display_dataframe_to_user("RF_Edge_Search_Results", df_rf_edge.sort_values("rmse").head(20))
display_dataframe_to_user("RF_Server_Search_Results", df_rf_server.sort_values("rmse").head(20))

with open("rf_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump({**rf_edge_cfg, **{k:best_rf_edge[k] for k in rf_edge_cfg["search_space"].keys()}}, f, indent=2, ensure_ascii=False)
with open("rf_server_final_config.json","w",encoding="utf-8") as f:
    json.dump({**rf_server_cfg, **{k:best_rf_server[k] for k in rf_server_cfg["search_space"].keys()}}, f, indent=2, ensure_ascii=False)

print("Best RF Edge:", best_rf_edge)
print("Best RF Server:", best_rf_server)


## 8) LSTM – Random Search (EDGE & SERVER)

In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from sklearn.preprocessing import MinMaxScaler

with open("config_lstm_edge_opt.json","r",encoding="utf-8") as f: lstm_edge_cfg = json.load(f)
with open("config_lstm_server_opt.json","r",encoding="utf-8") as f: lstm_server_cfg = json.load(f)

def scale_seq(X_tr_flat, X_te_flat, Y_tr, Y_te):
    xs = MinMaxScaler().fit(X_tr_flat)
    X_tr = xs.transform(X_tr_flat)
    X_te = xs.transform(X_te_flat)
    ys = MinMaxScaler().fit(Y_tr)
    ytr = ys.transform(Y_tr); yte = ys.transform(Y_te)
    return X_tr, X_te, ytr, yte, ys

F_srv = int(X2d_srv_tr.shape[1] / LAGS)
X3d_srv_tr = X2d_srv_tr.reshape(-1, LAGS, F_srv)
X3d_srv_te = X2d_srv_te.reshape(-1, LAGS, F_srv)
X3d_edge_tr = X2d_edge_tr.reshape(-1, LAGS, int(X2d_edge_tr.shape[1]/LAGS))
X3d_edge_te = X2d_edge_te.reshape(-1, LAGS, int(X2d_edge_te.shape[1]/LAGS))

def build_lstm(input_shape, num_layers=1, initial_units=64, dropout=0.1, horizon=HORIZON):
    m = Sequential()
    units = int(initial_units)
    for i in range(num_layers):
        m.add(LSTM(units, return_sequences=(i < num_layers-1), input_shape=input_shape if i==0 else None))
        m.add(Dropout(dropout)); m.add(BatchNormalization())
        units = max(units // 2, 4)
    m.add(Dense(horizon, activation="linear"))
    return m

def make_optimizer(p):
    name = str(p.get("optimizer","adam")).lower()
    lr = float(p.get("learning_rate", 1e-3))
    clipnorm = float(p.get("clipnorm", 0.0)) if p.get("clipnorm", 0.0) else None
    kw = {"learning_rate": lr}; 
    if clipnorm and clipnorm>0: kw["clipnorm"] = clipnorm
    if name=="adam": return Adam(**kw)
    if name=="rmsprop": return RMSprop(**kw)
    if name=="nadam": return Nadam(**kw)
    if name=="adamw":
        try: return tf.keras.optimizers.AdamW(weight_decay=float(p.get("weight_decay",0.0)), **kw)
        except: return Adam(**kw)
    return Adam(**kw)

def lstm_random_search(space, X3d_tr, Y_tr, X3d_te, Y_te, trials=20):
    X3d_tr_flat = X3d_tr.reshape(X3d_tr.shape[0], -1)
    X3d_te_flat = X3d_te.reshape(X3d_te.shape[0], -1)
    X_tr_s, X_te_s, Y_tr_s, Y_te_s, y_scaler = scale_seq(X3d_tr_flat, X3d_te_flat, Y_tr, Y_te)
    X3d_tr_s = X_tr_s.reshape(X3d_tr.shape); X3d_te_s = X_te_s.reshape(X3d_te.shape)
    results, best = [], None
    input_shape = (X3d_tr.shape[1], X3d_tr.shape[2])
    for _ in range(trials):
        p = sample_from_space(space)
        model = build_lstm(input_shape, p.get("num_layers",1), p.get("initial_units",64), p.get("dropout",0.1), HORIZON)
        opt = make_optimizer(p)
        model.compile(optimizer=opt, loss=p.get("loss","mse"), metrics=["mae"])
        split = int(len(X3d_tr_s) * (1 - VAL_FRACTION))
        X_fit, X_val = X3d_tr_s[:split], X3d_tr_s[split:]
        y_fit, y_val = Y_tr_s[:split], Y_tr_s[split:]
        cb=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
        model.fit(X_fit, y_fit, validation_data=(X_val, y_val) if len(X_val)>0 else None,
                  epochs=int(p.get("epochs",40)), batch_size=int(p.get("batch_size",32)), verbose=0, callbacks=cb)
        pred_s = model.predict(X3d_te_s, verbose=0)
        pred = y_scaler.inverse_transform(pred_s)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]: best = row
    return pd.DataFrame(results), best

df_lstm_edge, best_lstm_edge = lstm_random_search(lstm_edge_cfg["search_space"], X3d_edge_tr, Y_edge_tr, X3d_edge_te, Y_edge_te, trials=20)
df_lstm_server, best_lstm_server = lstm_random_search(lstm_server_cfg["search_space"], X3d_srv_tr, Y_srv_tr, X3d_srv_te, Y_srv_te, trials=30)

display_dataframe_to_user("LSTM_Edge_Search_Results", df_lstm_edge.sort_values("rmse").head(20))
display_dataframe_to_user("LSTM_Server_Search_Results", df_lstm_server.sort_values("rmse").head(20))

with open("lstm_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump({**lstm_edge_cfg, **{k:best_lstm_edge[k] for k in lstm_edge_cfg["search_space"].keys() if k in best_lstm_edge}}, f, indent=2, ensure_ascii=False)
with open("lstm_server_final_config.json","w",encoding="utf-8") as f:
    json.dump({**lstm_server_cfg, **{k:best_lstm_server[k] for k in lstm_server_cfg["search_space"].keys() if k in best_lstm_server}}, f, indent=2, ensure_ascii=False)

print("Best LSTM Edge:", best_lstm_edge)
print("Best LSTM Server:", best_lstm_server)


## 9) XGBoost – Random Search (EDGE & SERVER)

In [ ]:

from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

with open("config_xgb_edge_opt.json","r",encoding="utf-8") as f: xgb_edge_cfg = json.load(f)
with open("config_xgb_server_opt.json","r",encoding="utf-8") as f: xgb_server_cfg = json.load(f)

def xgb_random_search(space, X_tr, Y_tr, X_te, Y_te, trials=40):
    results, best = [], None
    for _ in range(trials):
        p = sample_from_space(space)
        base = XGBRegressor(
            n_estimators=int(p["n_estimators"]), max_depth=int(p["max_depth"]),
            learning_rate=float(p["learning_rate"]), subsample=float(p["subsample"]),
            colsample_bytree=float(p["colsample_bytree"]), min_child_weight=int(p["min_child_weight"]),
            gamma=float(p["gamma"]), reg_lambda=float(p["reg_lambda"]), reg_alpha=float(p["reg_alpha"]),
            tree_method=p["tree_method"], n_jobs=p["n_jobs"], random_state=p["random_state"],
            objective="reg:squarederror",
        )
        model = MultiOutputRegressor(base)
        model.fit(X_tr, Y_tr)
        pred = model.predict(X_te)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_xgb_edge, best_xgb_edge = xgb_random_search(xgb_edge_cfg["search_space"], X2d_edge_tr, Y_edge_tr, X2d_edge_te, Y_edge_te, trials=40)
df_xgb_server, best_xgb_server = xgb_random_search(xgb_server_cfg["search_space"], X2d_srv_tr, Y_srv_tr, X2d_srv_te, Y_srv_te, trials=60)

display_dataframe_to_user("XGB_Edge_Search_Results", df_xgb_edge.sort_values("rmse").head(20))
display_dataframe_to_user("XGB_Server_Search_Results", df_xgb_server.sort_values("rmse").head(20))

with open("xgb_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump({**xgb_edge_cfg, **{k:best_xgb_edge[k] for k in xgb_edge_cfg["search_space"].keys()}}, f, indent=2, ensure_ascii=False)
with open("xgb_server_final_config.json","w",encoding="utf-8") as f:
    json.dump({**xgb_server_cfg, **{k:best_xgb_server[k] for k in xgb_server_cfg["search_space"].keys()}}, f, indent=2, ensure_ascii=False)

print("Best XGB Edge:", best_xgb_edge)
print("Best XGB Server:", best_xgb_server)


## 10) LightGBM – Random Search (EDGE & SERVER)

In [ ]:

from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

with open("config_lgbm_edge_opt.json","r",encoding="utf-8") as f: lgbm_edge_cfg = json.load(f)
with open("config_lgbm_server_opt.json","r",encoding="utf-8") as f: lgbm_server_cfg = json.load(f)

def lgbm_random_search(space, X_tr, Y_tr, X_te, Y_te, trials=40):
    results, best = [], None
    for _ in range(trials):
        p = sample_from_space(space)
        base = LGBMRegressor(
            n_estimators=int(p["n_estimators"]), learning_rate=float(p["learning_rate"]),
            num_leaves=int(p["num_leaves"]), min_child_samples=int(p["min_child_samples"]),
            subsample=float(p["subsample"]), colsample_bytree=float(p["colsample_bytree"]),
            reg_alpha=float(p["reg_alpha"]), reg_lambda=float(p["reg_lambda"]),
            max_bin=int(p["max_bin"]), random_state=42, n_jobs=-1
        )
        model = MultiOutputRegressor(base)
        model.fit(X_tr, Y_tr)
        pred = model.predict(X_te)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_lgbm_edge, best_lgbm_edge = lgbm_random_search(lgbm_edge_cfg["search_space"], X2d_edge_tr, Y_edge_tr, X2d_edge_te, Y_edge_te, trials=40)
df_lgbm_server, best_lgbm_server = lgbm_random_search(lgbm_server_cfg["search_space"], X2d_srv_tr, Y_srv_tr, X2d_srv_te, Y_srv_te, trials=60)

display_dataframe_to_user("LGBM_Edge_Search_Results", df_lgbm_edge.sort_values("rmse").head(20))
display_dataframe_to_user("LGBM_Server_Search_Results", df_lgbm_server.sort_values("rmse").head(20))

with open("lgbm_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump({**lgbm_edge_cfg, **{k:best_lgbm_edge[k] for k in lgbm_edge_cfg["search_space"].keys()}}, f, indent=2, ensure_ascii=False)
with open("lgbm_server_final_config.json","w",encoding="utf-8") as f:
    json.dump({**lgbm_server_cfg, **{k:best_lgbm_server[k] for k in lgbm_server_cfg["search_space"].keys()}}, f, indent=2, ensure_ascii=False)

print("Best LGBM Edge:", best_lgbm_edge)
print("Best LGBM Server:", best_lgbm_server)


## 11) CNN1D – Random Search (EDGE & SERVER)

In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, Dropout, GlobalAveragePooling1D, Dense, Input
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from sklearn.preprocessing import MinMaxScaler

with open("config_cnn1d_edge_opt.json","r",encoding="utf-8") as f: cnn_edge_cfg = json.load(f)
with open("config_cnn1d_server_opt.json","r",encoding="utf-8") as f: cnn_server_cfg = json.load(f)

def build_cnn1d(input_shape, blocks=2, base_filters=64, kernel_size=5, dropout=0.1, activation="relu", horizon=HORIZON):
    m = Sequential([Input(shape=input_shape)])
    filters = int(base_filters)
    for i in range(int(blocks)):
        m.add(Conv1D(filters=filters, kernel_size=int(kernel_size), padding="same"))
        m.add(BatchNormalization()); m.add(Activation(activation))
        if dropout and dropout>0: m.add(Dropout(float(dropout)))
        filters = max(filters // 2, 4)
    m.add(GlobalAveragePooling1D())
    m.add(Dense(max(filters, 8))); m.add(Activation(activation))
    m.add(Dense(horizon, activation="linear"))
    return m

def make_optimizer(p):
    name = str(p.get("optimizer","adam")).lower()
    lr = float(p.get("learning_rate", 1e-3))
    clipnorm = float(p.get("clipnorm", 0.0)) if p.get("clipnorm", 0.0) else None
    kw = {"learning_rate": lr}; 
    if clipnorm and clipnorm>0: kw["clipnorm"] = clipnorm
    if name=="adam": return Adam(**kw)
    if name=="rmsprop": return RMSprop(**kw)
    if name=="nadam": return Nadam(**kw)
    if name=="adamw":
        try: return tf.keras.optimizers.AdamW(weight_decay=float(p.get("weight_decay",0.0)), **kw)
        except: return Adam(**kw)
    return Adam(**kw)

# SERVER reshape & scale
F_srv = int(X2d_srv_tr.shape[1] / LAGS)
X3d_srv_tr = X2d_srv_tr.reshape(-1, LAGS, F_srv)
X3d_srv_te = X2d_srv_te.reshape(-1, LAGS, F_srv)
X3d_srv_tr_flat = X3d_srv_tr.reshape(X3d_srv_tr.shape[0], -1)
X3d_srv_te_flat = X3d_srv_te.reshape(X3d_srv_te.shape[0], -1)
xs_srv = MinMaxScaler().fit(X3d_srv_tr_flat)
X3d_srv_tr_s = xs_srv.transform(X3d_srv_tr_flat).reshape(X3d_srv_tr.shape)
X3d_srv_te_s = xs_srv.transform(X3d_srv_te_flat).reshape(X3d_srv_te.shape)
ys_srv = MinMaxScaler().fit(Y_srv_tr)
Y_srv_tr_s = ys_srv.transform(Y_srv_tr); Y_srv_te_s = ys_srv.transform(Y_srv_te)

# EDGE reshape & scale
F_edge = int(X2d_edge_tr.shape[1] / LAGS)
X3d_edge_tr = X2d_edge_tr.reshape(-1, LAGS, F_edge)
X3d_edge_te = X2d_edge_te.reshape(-1, LAGS, F_edge)
X3d_edge_tr_flat = X3d_edge_tr.reshape(X3d_edge_tr.shape[0], -1)
X3d_edge_te_flat = X3d_edge_te.reshape(X3d_edge_te.shape[0], -1)
xs_edge = MinMaxScaler().fit(X3d_edge_tr_flat)
X3d_edge_tr_s = xs_edge.transform(X3d_edge_tr_flat).reshape(X3d_edge_tr.shape)
X3d_edge_te_s = xs_edge.transform(X3d_edge_te_flat).reshape(X3d_edge_te.shape)
ys_edge = MinMaxScaler().fit(Y_edge_tr)
Y_edge_tr_s = ys_edge.transform(Y_edge_tr); Y_edge_te_s = ys_edge.transform(Y_edge_te)

def cnn_random_search(space, X3d_tr_s, Y_tr_s, X3d_te_s, Y_te, y_scaler, trials=20):
    results, best = [], None
    input_shape = (X3d_tr_s.shape[1], X3d_tr_s.shape[2])
    for _ in range(trials):
        p = sample_from_space(space)
        model = build_cnn1d(input_shape, p.get("cnn_blocks",2), p.get("cnn_base_filters",64),
                            p.get("cnn_kernel_size",5), p.get("cnn_dropout",0.1),
                            p.get("cnn_activation","relu"), HORIZON)
        opt = make_optimizer(p)
        model.compile(optimizer=opt, loss="huber", metrics=["mae"])
        split = int(len(X3d_tr_s) * (1 - VAL_FRACTION))
        X_fit, X_val = X3d_tr_s[:split], X3d_tr_s[split:]
        y_fit, y_val = Y_tr_s[:split], Y_tr_s[split:]
        cb=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
        model.fit(X_fit, y_fit, validation_data=(X_val, y_val) if len(X_val)>0 else None,
                  epochs=int(p.get("epochs",40)), batch_size=int(p.get("batch_size",32)), verbose=0, callbacks=cb)
        pred_s = model.predict(X3d_te_s, verbose=0)
        pred = y_scaler.inverse_transform(pred_s)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_cnn_edge, best_cnn_edge = cnn_random_search(cnn_edge_cfg["search_space"], X3d_edge_tr_s, Y_edge_tr_s, X3d_edge_te_s, Y_edge_te, ys_edge, trials=20)
df_cnn_server, best_cnn_server = cnn_random_search(cnn_server_cfg["search_space"], X3d_srv_tr_s, Y_srv_tr_s, X3d_srv_te_s, Y_srv_te, ys_srv, trials=30)

display_dataframe_to_user("CNN1D_Edge_Search_Results", df_cnn_edge.sort_values("rmse").head(20))
display_dataframe_to_user("CNN1D_Server_Search_Results", df_cnn_server.sort_values("rmse").head(20))

with open("cnn1d_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump({**cnn_edge_cfg, **{k:best_cnn_edge[k] for k in cnn_edge_cfg["search_space"].keys() if k in best_cnn_edge}}, f, indent=2, ensure_ascii=False)
with open("cnn1d_server_final_config.json","w",encoding="utf-8") as f:
    json.dump({**cnn_server_cfg, **{k:best_cnn_server[k] for k in cnn_server_cfg["search_space"].keys() if k in best_cnn_server}}, f, indent=2, ensure_ascii=False)

print("Best CNN Edge:", best_cnn_edge)
print("Best CNN Server:", best_cnn_server)
